In [1]:
import numpy as np
from levisim import *



# the fortran is set up to save spectral values in the 0-400 kHz range
root = '../fortran/'

# generate these data files by running
# gfortran -O2 -llapack NParticle.f90  3DEQRom.f90 ParametersNpair.f90 Suscept.f90 PSD.f90 NoiseVec2D.f90; ./a.out
# inside the ../fortran directory
XYZ_file = root + 'FTanXYZ.dat'
CBM_file = root +'FTCBM.dat'
OPT_file = root + 'FTanOPT.dat'

xyz_data = np.loadtxt(XYZ_file)
cbm_data = np.loadtxt(CBM_file)
opt_data = np.loadtxt(OPT_file)

# kHz step size of fortran simulation
freq_step_size = xyz_data[:, 0][1] - xyz_data[:, 0][0]
print(f'{freq_step_size=} kHz')

# print all the Fortran params
# NOTE: not all of these have to match
with open(root + 'CSCAVITY.h', 'r') as file:
    fortran_params = file.read()
print(fortran_params)


freq_step_size=0.5 kHz
PARAMETER(&
    ! ---------------------------------------------------------------------------------------------
    ! constants
    pi=3.141592653589793d0, &
    hbar=1.0545718176461565d-34, & ! J/(rad/s)
    speed_of_light=299792458.d0, & ! m/s
    kB=1.380649d-23, & ! J/K
    vacuum_permittivity=8.8541878128d-12, & ! F/m
    ! ---------------------------------------------------------------------------------------------
    ! units
    Hz=pi + pi, &
    kHz=1.d3 * Hz, &
    ! ---------------------------------------------------------------------------------------------
    ! tweezer
    tweezer_power=0.4d0, & ! Watts
    ! angle between tweezer polarization and cavity axis, as fraction of π
    ! 0.25 => π/4 = 45 degrees
    theta_tweezer_over_pi=0.25d0, &
    tweezer_waist_X=1.0 * 0.85d-6, &
    tweezer_waist_Y=1.25 * 0.85d-6, &
    ! ---------------------------------------------------------------------------------------------
    ! bead
    bead_radius=75.d-9, 

In [2]:
# NOTE: these toy_params and kwargs are the only things needed to match the Fortran
toy_params = {
    'model': 'pairs',
    'first_Ωx': 312*kHz,
    'Ω_gap': 15*kHz,
    'Ωz_to_Ωx_multiplier': 10,

    'toy_gx': -40*kHz,
    'toy_gy': -40*kHz,

    'use_Γ_formula': True,
}

cavity_kwargs = {
    'Δ': -450*kHz,
    'L': 0.96*cm,
    'Finesse': 40_000,
    'θhom': π/4,
}

gas_kwargs = {
    'P': 1e-6*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}

N, ax = 8, 2
ω_low, ω_high, ω_step = 0, 400, freq_step_size
mph = MultiPhysics(
    num_traps=N,
    Cartesian_axes_per_trap=ax,
    ω_range=np.arange(ω_low, ω_high, ω_step) * kHz,

    cavity_kwargs=cavity_kwargs,
    gas_kwargs=gas_kwargs,
    toy_params=toy_params,
)
mph.cavity.info
mph.info
mph.info_per_dimension

methods = [
    QuantumLinearTheoryMethod.LINEAR_AMPLIFIER_MODEL,
    QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION,
]
PSDs, homodyne, heterodyne = {}, {}, {}
for method in methods:
    PSDs[method] =  mph.calculate_all_hybridised_position_spectra(method)
    homodyne[method] = mph.calculate_homodyne_spectrum(method)
    heterodyne[method] = mph.calculate_heterodyne_spectrum(method)


Cavity(id=5441172240
    Length = 0.96000 cm
    waist = 61.00000 μm
    Finesse = 4.000e+04
    Δ/2π = -450.00000 kHz

    κ/2π = 390.35476 kHz
    Volume = 2.80557e-11 m³
    ω/2π = 281.75983 THz
    k = 5.90525e+06 rad/m
    λ = 1.06400e-06 m
    xR = 1.09867e-02 m
    εcav = 1.93851e+01 V/m

    θhom = 45.00000 °
    Ωhet = 0.00000 kHz
)
MultiPhysics(
    num_traps = 8
    Cartesian_axes_per_trap = 2
    ω_range = [0.0, 399.5] kHz
    num_ω_points = 800

    setup_hybridisation_power_series = False
    compare_Rjk_calculations = False

    id(gas) = 4355586384
    id(cavity) = 5441172240
)
toy_params = {
    type = pairs
    first_Ωx = 312.0 kHz
    Ω_gap = 15.0 kHz
    Ωz_to_Ωx_multiplier = 10.0

    ρ = 2200.0 kg/m³
    r = 75.0 nm
    m = 3.88772e-18 kg

    P = 0.0 mbar
    T = 300.0 K
    〈v〉 = 500.0 m/s

    all Ωx = [312. 297. 282. 267. 252. 237. 222. 207. 192.] kHz
    all Ωy = [313.5 298.5 283.5 268.5 253.5 238.5 223.5 208.5] kHz
    all Ωz = [31.2 29.7 28.2 26.7 25.2 23.7

In [3]:
xyz_data[:, 0][:5]

array([0.5, 1. , 1.5, 2. , 2.5])

In [4]:
mph.ω_range[:5] / kHz

array([0. , 0.5, 1. , 1.5, 2. ])

In [5]:
num_samples_to_compare = 3
print(f'First {num_samples_to_compare} X1 PSD values...')
print(xyz_data[:, 1][:num_samples_to_compare], 'method=fortran')
for method in methods:
    print(PSDs[method][:, 0][1:num_samples_to_compare + 1], f'{method=}')

print()
print(f'First {num_samples_to_compare} homodyne values...')
print(opt_data[:, 2][:num_samples_to_compare], 'method=fortran')
for method in methods:
    print(homodyne[method][1:num_samples_to_compare + 1], f'{method=}')

print()
print(f'First {num_samples_to_compare} heterodyne values...')
print(opt_data[:, 1][:num_samples_to_compare], 'method=fortran')
for method in methods:
    print(heterodyne[method][1:num_samples_to_compare + 1], f'{method=}')

First 3 X1 PSD values...
[4.006651e-07 4.012007e-07 4.017570e-07] method=fortran
[4.00665084e-07 4.01200675e-07 4.01756939e-07] method=LAM
[4.00665084e-07 4.01200675e-07 4.01756939e-07] method=Numerical H

First 3 homodyne values...
[3.953034 3.953191 3.95345 ] method=fortran
[3.95303467 3.95319013 3.95344925] method=LAM
[3.95303467 3.95319013 3.95344925] method=Numerical H

First 3 heterodyne values...
[3.191611 3.192777 3.19408 ] method=fortran
[3.19161193 3.19277658 3.19407926] method=LAM
[3.19161193 3.19277658 3.19407926] method=Numerical H


In [8]:
num_samples_to_compare = len(opt_data[:, 1])

# fortran output goes to 7 significant figures
rtol = 1e-6
freqs = opt_data[:, 0]
for ii in range(num_samples_to_compare):

    freq = freqs[ii]
    for method in methods:

        # the first entries are off shifted off by one, hence ii and ii + 1
        for jj in range(1, xyz_data.shape[1]):
            fortran_PSD_value = xyz_data[:, jj][ii]
            python_PSD_value = PSDs[method][:, jj - 1][ii + 1]
            # the first column of xyz_data are the frequencies, hence jj and jj - 1
            assert np.isclose(fortran_PSD_value, python_PSD_value, rtol=rtol), f'{ii=}, {freq=}, {method=}, {fortran_PSD_value=}, {python_PSD_value=}'

        fortran_heterodyne_value = opt_data[:, 1][ii]
        python_heterodyne_value = heterodyne[method][ii + 1]
        assert np.isclose(fortran_heterodyne_value, python_heterodyne_value, rtol=rtol), f'{ii=}, {freq=}, {method=}, {fortran_heterodyne_value=}, {python_heterodyne_value=}'

        fortran_homodyne_value = opt_data[:, 2][ii]
        python_homodyne_value = homodyne[method][ii + 1]
        assert np.isclose(fortran_homodyne_value, python_homodyne_value, rtol=rtol), f'{ii=}, {freq=}, {method=}, {fortran_homodyne_value=}, {python_homodyne_value=}'